In [56]:
from datasets import load_dataset
ds = load_dataset("eugene6923/Self-Bench")

In [57]:
tags = ["colors", "color_attr", "position", "two_object"]

In [6]:
# # get unique models
# unique_models = set(filtered_ds["model"])
# print(f"Unique models: {unique_models}")

In [7]:
# # filter such that model='stable-diffusion-3-medium-diffusers
# sd3_ds = filtered_ds.filter(lambda x: x["model"] == "stable-diffusion-3-medium-diffusers")

In [ ]:
def get_samples_with_tag(tag, num_samples, ds=ds):
    """
    Get unique prompts from the dataset for a given tag.
    Includes the corresponding tag and class in the output.
    """
    tag_ds = ds.filter(lambda x: x["tag"] == tag)
    seen_prompts = set()
    count = 0
    samples = []
    for i in range(len(tag_ds)):
        if count >= num_samples:
            break
        prompt = tag_ds[i]["prompt"]
        if prompt not in seen_prompts:
            seen_prompts.add(prompt)
            samples.append({
                "prompt": prompt,
                "tag": tag,
                "class": tag_ds[i]["class"]
            })
            count += 1
    return samples

In [71]:
filtered_ds = ds["train"]
tags = ["colors", "color_attr", "position", "two_object"]

Filter: 100%|██████████| 6636/6636 [01:46<00:00, 62.26 examples/s] 


In [74]:
print(len(final_samples))

100


In [65]:
print(len(final_samples))
# filter the prompts which have only "left"
some_final_samples = [s for s in final_samples if "left" in s["prompt"]]

# some_final_samples = [s for s in final_samples if "left" in s["prompt"] or "right" in s["prompt"]]
print(f"Final samples count after filtering: {len(some_final_samples)}")

100
Final samples count after filtering: 19


In [62]:
print(len(final_samples))
# filter the prompts which have only "left"
some_final_samples = [s for s in final_samples if "left" in s["prompt"]]

# some_final_samples = [s for s in final_samples if "left" in s["prompt"] or "right" in s["prompt"]]
print(f"Final samples count after filtering: {len(some_final_samples)}")

100
Final samples count after filtering: 19


In [19]:
# For color_attr, add prompts where the colors are swapped
color_attr_samples = [s for s in final_samples if s["tag"] == "color_attr"]
swapped_samples = []
for sample in color_attr_samples:
    original_prompt = sample["prompt"]
    class_info = sample["class"]
    
    # "a photo of a green skis and a brown airplane"
    parts = original_prompt.split(" and ")
    if len(parts) == 2:
        first_part = parts[0].replace("a photo of a ", "")
        second_part = parts[1]
        
        first_color, first_object = first_part.split(" ", 1)
        second_color, second_object = second_part.split(" ", 1)
        
        swapped_prompt = f"a photo of a {second_color} {first_object} and a {first_color} {second_object}"
        
        swapped_samples.append({
            "prompt": swapped_prompt,
            "tag": "color_attr_swapped",
            "class": class_info,
            "original_prompt": original_prompt
        })

final_samples.extend(swapped_samples)
print(f"Total samples after adding color_attr_swapped: {len(final_samples)}")


Total samples after adding color_attr_swapped: 50


In [20]:
# Swap objects in two_object prompts
two_object_samples = [s for s in final_samples if s["tag"] == "two_object"]
swapped_two_object_samples = []
for sample in two_object_samples:
    original_prompt = sample["prompt"]
    class_info = sample["class"]
    
    # "a photo of a car and a bus"
    parts = original_prompt.split(" and ")
    if len(parts) == 2:
        first_part = parts[0].replace("a photo of a ", "")
        second_part = parts[1]
        
        first_object = first_part
        second_object = second_part
        
        swapped_prompt = f"a photo of a {second_object} and a {first_object}"
        
        swapped_two_object_samples.append({
            "prompt": swapped_prompt,
            "tag": "two_object_swapped",
            "class": class_info,
            "original_prompt": original_prompt
        })
        print(f"Swapped two_object prompt: {swapped_prompt}")

final_samples.extend(swapped_two_object_samples)
print(f"Total samples after adding two_object_swapped: {len(final_samples)}")

Swapped two_object prompt: a photo of a a sports ball and a bench
Swapped two_object prompt: a photo of a a snowboard and a toothbrush
Swapped two_object prompt: a photo of a an oven and a toaster
Swapped two_object prompt: a photo of a a vase and a broccoli
Swapped two_object prompt: a photo of a a wine glass and a tennis racket
Swapped two_object prompt: a photo of a a knife and a fork
Swapped two_object prompt: a photo of a a cake and a hair drier
Swapped two_object prompt: a photo of a a giraffe and a horse
Swapped two_object prompt: a photo of a a computer keyboard and a horse
Swapped two_object prompt: a photo of a a carrot and a toothbrush
Total samples after adding two_object_swapped: 60


In [21]:
# For "colors" tag, change the color to a different one
import random

color_samples = [s for s in final_samples if s["tag"] == "colors"]
color_options = ["red", "blue", "green", "yellow", "purple", "orange", "pink", "brown", "black", "white", "gray"]
modified_color_samples = []

for sample in color_samples:
    original_prompt = sample["prompt"]
    class_info = sample["class"]
    
    # "a photo of a red car"
    parts = original_prompt.split(" ")
    if len(parts) >= 5:
        current_color = parts[4]
        object_part = " ".join(parts[5:])
        
        new_color = current_color
        while new_color == current_color:
            new_color = random.choice(color_options)
        
        modified_prompt = f"a photo of a {new_color} {object_part}"
        
        modified_color_samples.append({
            "prompt": modified_prompt,
            "tag": "colors_modified",
            "class": class_info,
            "original_prompt": original_prompt
        })

final_samples.extend(modified_color_samples)
print(f"Total samples after adding colors_modified: {len(final_samples)}")

Total samples after adding colors_modified: 70


In [27]:
# For tag="position", add prompts where the
#   1. objects are swapped
#   2. positional words are made opposite or changed

# 1. add prompts where the objects are swapped
position_samples = [s for s in final_samples if s["tag"] == "position"]
swapped_position_samples = []
for sample in position_samples:
    original_prompt = sample["prompt"]
    class_info = sample["class"]
    
    # "a photo of a car to the left of a bus" or "a photo of a cat above a dog"
    parts = original_prompt.split(" ")
    first_object = parts[4]
    second_object = parts[-1]
    positional_part = " ".join(parts[5:-2])  # e.g., "to the left of" or "above"
    swapped_prompt = f"a photo of a {second_object} {positional_part} a {first_object}"
    swapped_position_samples.append({
        "prompt": swapped_prompt,
        "tag": "position_swapped",
        "class": class_info,
        "original_prompt": original_prompt  
    })

final_samples.extend(swapped_position_samples)
print(f"Total samples after adding position_swapped: {len(final_samples)}")

Total samples after adding position_swapped: 80


In [34]:
# 2. change positional words to their opposites or simething else


In [35]:
# # 2. change positional words to their opposites or similar
# # using huggingface llms to change positional words in the prompt

# # use llm to get opposite positional words
# def get_opposite_position(word):
#     pass


# # add new samples with changed positional words
# changed_position_samples = []
# for sample in position_samples:
#     original_prompt = sample["prompt"]
#     print(original_prompt)

# #     class_info = sample["class"]
    
# #     # "a photo of a car to the left of a bus" or "a photo of a fork above a hair drier"
# #     parts = original_prompt.split(" ")
# #     positional_part = " ".join(parts[5:-2])
# #     opposite_word = get_opposite_position(positional_part)
# #     if opposite_word and opposite_word != positional_part:
# #         modified_prompt = original_prompt.replace(positional_part, opposite_word)
# #         print(f"Original position prompt: {original_prompt}")
# #         print(f"Modified position prompt: {modified_prompt}")
# #         changed_position_samples.append({
# #             "prompt": modified_prompt,
# #             "tag": "position_modified",
# #             "class": class_info,
# #             "original_prompt": original_prompt
# #         })

# # # final_samples.extend(changed_position_samples)
# # # print(f"Total samples after adding position_modified: {len(final_samples)}")

In [36]:
# save final_samples as csv
import pandas as pd
df = pd.DataFrame(final_samples)
df.to_csv("final_samples_sd3.csv", index=False)